# 08 — Model: PyTorch Neural Net

Requires `train_features.csv` / `test_features.csv` from **01_feature_engineering.ipynb**.

Run `!pip install torch` once if you don't already have it.

A small feed-forward net: learned embeddings for the 3 categorical columns concatenated with the
standardized numeric features, then a few dense layers with BatchNorm + Dropout. Trees dominate on this
kind of tabular data, so don't expect the NN to beat the GBMs solo — its value is diversity for the blend
(it makes very different kinds of errors than a tree model). Saves `oof_nn.csv` and `test_pred_nn.csv` for
the ensembling notebook.

In [1]:
!pip install -q torch


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

DATA_DIR = "."   # <-- folder with train_features.csv / test_features.csv from notebook 01
N_FOLDS = 5
SEED = 42

train_fe = pd.read_csv(f"{DATA_DIR}/train_features.csv")
test_fe = pd.read_csv(f"{DATA_DIR}/test_features.csv")

num_cols = ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours',
            'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time',
            'notif_per_hour', 'app_per_hour', 'mins_per_appopen', 'mins_per_notif', 'productivity',
            'sleep_screen_sum', 'nonscreen_hours', 'screen_plus_weekend', 'screen_sleep_ratio',
            'sm_ratio', 'game_ratio', 'work_ratio', 'screen_minus_work', 'weekday_weekend_ratio',
            'screen_x_sm', 'screen_x_weekend', 'sm_x_weekend', 'screen_x_sleep']
cat_cols = ['gender', 'stress_level', 'academic_work_impact']
feat_cols = num_cols + cat_cols

X = train_fe[feat_cols].copy()
Xtest = test_fe[feat_cols].copy()
y = train_fe['addicted_label'].values

for c in cat_cols:
    X[c] = X[c].astype('category')
    Xtest[c] = Xtest[c].astype('category')

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))
print(X.shape, Xtest.shape)

from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import time

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch device:", DEVICE)


(691369, 30) (296302, 30)
torch device: cpu


In [3]:
cat_cardinalities = {c: int(pd.concat([X[c], Xtest[c]]).astype('category').cat.codes.max()) + 2 for c in cat_cols}

class TabularNN(nn.Module):
    def __init__(self, n_num, cat_cardinalities, emb_dim=4, hidden=(128, 64)):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(card, emb_dim) for card in cat_cardinalities.values()])
        in_dim = n_num + emb_dim * len(cat_cardinalities)
        layers = []
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(0.2)]
            in_dim = h
        layers += [nn.Linear(in_dim, 1)]
        self.mlp = nn.Sequential(*layers)

    def forward(self, x_num, x_cat):
        embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.cat([x_num] + embs, dim=1)
        return self.mlp(x).squeeze(1)

# prep numeric (median-impute + standardize) and categorical (integer codes, NaN -> own code) arrays once
X_num_raw = X[num_cols].fillna(X[num_cols].median())
Xtest_num_raw = Xtest[num_cols].fillna(X[num_cols].median())
scaler = StandardScaler().fit(X_num_raw)
X_num_all = scaler.transform(X_num_raw).astype(np.float32)
Xtest_num_all = scaler.transform(Xtest_num_raw).astype(np.float32)

X_cat_all = np.zeros((len(X), len(cat_cols)), dtype=np.int64)
Xtest_cat_all = np.zeros((len(Xtest), len(cat_cols)), dtype=np.int64)
for i, c in enumerate(cat_cols):
    codes, uniques = pd.factorize(pd.concat([X[c], Xtest[c]]))
    X_cat_all[:, i] = codes[:len(X)] + 1   # shift so missing (-1) -> 0
    Xtest_cat_all[:, i] = codes[len(X):] + 1


In [ ]:
oof_nn = np.zeros(len(X))
test_nn = np.zeros(len(Xtest))

Xtest_num_t = torch.tensor(Xtest_num_all, dtype=torch.float32).to(DEVICE)
Xtest_cat_t = torch.tensor(Xtest_cat_all, dtype=torch.long).to(DEVICE)

t0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(folds):
    model = TabularNN(len(num_cols), cat_cardinalities).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    loss_fn = nn.BCEWithLogitsLoss()

    train_ds = TensorDataset(
        torch.tensor(X_num_all[tr_idx]), torch.tensor(X_cat_all[tr_idx]),
        torch.tensor(y[tr_idx], dtype=torch.float32)
    )
    train_loader = DataLoader(train_ds, batch_size=4096, shuffle=True)

    Xva_num = torch.tensor(X_num_all[va_idx], dtype=torch.float32).to(DEVICE)
    Xva_cat = torch.tensor(X_cat_all[va_idx], dtype=torch.long).to(DEVICE)

    best_auc, best_state, patience, bad_epochs = 0, None, 5, 0
    for epoch in range(50):
        model.train()
        for xb_num, xb_cat, yb in train_loader:
            xb_num, xb_cat, yb = xb_num.to(DEVICE), xb_cat.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            out = model(xb_num, xb_cat)
            loss = loss_fn(out, yb)
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            p_va = torch.sigmoid(model(Xva_num, Xva_cat)).cpu().numpy()
        auc = roc_auc_score(y[va_idx], p_va)
        if auc > best_auc:
            best_auc, best_state, bad_epochs = auc, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        p_va = torch.sigmoid(model(Xva_num, Xva_cat)).cpu().numpy()
        p_test = torch.sigmoid(model(Xtest_num_t, Xtest_cat_t)).cpu().numpy()
    oof_nn[va_idx] = p_va
    test_nn += p_test / N_FOLDS
    print(f"fold {fold} auc={roc_auc_score(y[va_idx], p_va):.5f}  ({time.time()-t0:.0f}s elapsed)")

print("Neural Net OOF AUC:", roc_auc_score(y, oof_nn))


fold 0 auc=0.93880  (430s elapsed)


In [ ]:
pd.DataFrame({'id': train_fe['id'], 'oof_pred': oof_nn}).to_csv(f"{DATA_DIR}/oof_nn.csv", index=False)
pd.DataFrame({'id': test_fe['id'], 'test_pred': test_nn}).to_csv(f"{DATA_DIR}/test_pred_nn.csv", index=False)
print("saved oof_nn.csv and test_pred_nn.csv")
